# 🚀 H5-OmniFusion: Phase 4 Mastery Notebook
**Multi-Modal Depression Detection Clinical Validation Suite**

This notebook automates the entire Phase 4 workflow:
1.  **Persistent Setup**: Installs heavy libraries (Mamba) once and saves them to your Drive as a ZIP for instant loading later.
2.  **5-Fold Validation**: Trains the Medium Tier model across all 5 splits.
3.  **Wisdom of the Crowd**: Runs the Soft-Voting Ensemble using all 5 fold-checkpoints.
4.  **Scientific Ablation**: Measures the importance of Text, Audio, and Video.

## 🛠️ Step 1: Environment Sync & Persistent Cache
This cell checks your Google Drive for a pre-built environment ZIP. 
- **First Run**: Takes ~10-15 mins to build Mamba.
- **Subsequent Runs**: Takes ~30 seconds (installs from ZIP).

In [ ]:
import os, sys, shutil
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Configuration
REPO_URL = "https://github.com/nithin12342/phase2.git"
DRIVE_BASE = "/content/drive/MyDrive/DAIC-WOZ_Datasets"
CACHE_ZIP = f"{DRIVE_BASE}/mamba_env_v1.zip"
LOCAL_PKGS = "/content/local_pkgs"

# 3. Clone / Sync Project Code
%cd /content/
if os.path.exists('/content/phase2'):
    !rm -rf /content/phase2
!git clone {REPO_URL} /content/phase2

# 4. Setup Persistent Environment
os.makedirs(LOCAL_PKGS, exist_ok=True)
if os.path.exists(CACHE_ZIP):
    print("📦 Found cached environment on Drive! Unzipping for speed...")
    !unzip -q {CACHE_ZIP} -d {LOCAL_PKGS}
    print("✅ Environment restored in seconds.")
else:
    print("⏳ No cache found. Performing one-time build (Mamba-SSM)... This may take 12-15 mins.")
    !apt-get install -y ninja-build
    !pip install ninja packaging
    # Install to local folder fast
    !MAX_JOBS=4 pip install --target={LOCAL_PKGS} mamba-ssm causal-conv1d>=1.4.0 --no-build-isolation
    !pip install --target={LOCAL_PKGS} transformers opensmile librosa
    
    print("💾 Backing up environment to Drive for future 5-second loading...")
    %cd {LOCAL_PKGS}
    !zip -r {CACHE_ZIP} .
    %cd /content/
    print(f"✅ Cache saved to {CACHE_ZIP}")

# 5. Link Paths
sys.path.insert(0, LOCAL_PKGS)
os.environ['PYTHONPATH'] = f"{LOCAL_PKGS}:/content/phase2/ml_pipeline/h5_omnifusion"
print("\n🚀 READY! Path and Dependencies Synced.")

## 🚀 Step 2: 5-Fold Clinical Re-Training
We will train the **Medium Tier** (12M params) across 5 folds. These models will save directly to your Drive.

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase4"
OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
ENV_PKGS = "/content/local_pkgs"
!mkdir -p {SAVE_DIR}

for fold in range(5):
    print(f"\n{'='*40}\n🔥 STARTING FOLD {fold}/5\n{'='*40}")
    !PYTHONPATH={OS_PATH}:{ENV_PKGS} python {OS_PATH}/scripts/train.py \
        --data_dir "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output" \
        --labels_csv "/content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv" \
        --tier medium \
        --fold {fold} \
        --epochs 25 \
        --lr 1e-4 \
        --output_dir {SAVE_DIR}

print("\n✅ All 5 Folds Training Complete!")

## 🗳️ Step 3: Ensemble Prediction (Wisdom of the Crowd)
We combine the 5 models to push the F1 score towards **0.60+**.

In [ ]:
import os
OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
ENV_PKGS = "/content/local_pkgs"

# 0. Sync latest bugfixes from GitHub
%cd /content/phase2/
!git pull origin main
%cd /content/

# 1. Run Ensemble with explicit path and global input for full coverage (358 samples)
print("🗳️ Running Ensemble Prediction across all datasets...")
!PYTHONPATH={OS_PATH}:{ENV_PKGS} python {OS_PATH}/scripts/ensemble_predict.py \
    --checkpoints /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase4 \
    --input "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/" \
    --tier medium \
    --output "/content/final_ensemble_results.csv"

print("\n--- FINAL EVALUATION ---")
!PYTHONPATH={OS_PATH}:{ENV_PKGS} python {OS_PATH}/scripts/evaluate_ensemble.py

## 🔬 Step 4: Modality Ablation (Scientific Proof)
Run this to automatically identify which modality (Text, Audio, Video) is the most critical for the diagnosis.

In [ ]:
OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
ENV_PKGS = "/content/local_pkgs"
!PYTHONPATH={OS_PATH}:{ENV_PKGS} python {OS_PATH}/scripts/run_ablation_study.py